# Sandbox del pipeline sin malla -- 100% auto-contenido

Corre TODOS los pasos del pipeline sin malla **excepto el renderizado**
(los renders ya existen en `data/renders/`, se copian, no se regeneran) sobre
un subconjunto chico de 9 objetos (3 BUENO / 3 MEDIO / 3 MALO por
`angular_error` con `axis_v06_nomesh`):

```
1. Copiar .obj/.txt (GT) y renders existentes -> carpeta propia en Experiments/
2. Prompts (texto editable en una celda, no en archivos .txt aparte)
3. Geometria: camara, triangulacion, filtro fondo/objeto (todo inline)
4. Molmo2: carga del modelo + inferencia (inline)          [necesita GPU]
5. Correr inferencia sobre los 9 objetos
6. Estimar eje sin malla (triangulacion, inline)
7. Evaluar contra GT (inline)
8. Resultados por objeto, con su categoria BUENO/MEDIO/MALO
9. Visualizacion 2D/3D (opcional)
```

**Ningun paso llama a un script .py del repo** -- todo el codigo (prompts,
geometria de camara/triangulacion, parseo de la salida de Molmo2, metricas)
esta copiado inline en las celdas de abajo, para poder editarlo directo ahi
y probar variaciones (de prompt, de post-procesamiento, de metrica) sin salir
del notebook. Es una copia funcional de la logica de
`MolmoPointing/molmo_multiview_runner.py`, `pipeline_common/camera.py`,
`pipeline_common/triangulation.py`, `Mapping/estimate_symmetry_no_mesh.py` y
`Mapping/evaluate.py` -- si mas adelante queres llevar una variante ganadora
al pipeline real, hay que trasplantarla a mano a esos archivos.

**No incluye `SDE_ref`/`F1_ref`** (necesitan `gpytoolbox` + muestreo de
superficie -- mas pesado) -- las metricas que si trae (`angular_error`,
`translation_error`, `precision@theta`) alcanzan para comparar variantes
rapido; agregalas a mano si las necesitas.

Los `.obj`/`.txt` y los renders (PNG + `manifest.json` + `metadata_all.json`)
se **copian** desde `data/` (solo lectura, nunca se modifican) -- todo lo
demas (predicciones, ejes estimados, metricas) se genera de cero dentro del
sandbox cada vez que corres las celdas.

## 0. Setup

In [ ]:
import json
import re
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import trimesh
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

REPO_ROOT = Path(r"C:\Users\HP\Desktop\Seminario de Tesis I\Symmetry-Detection-Using-Multimodal-Vision-Language-Models")
DATA_ROOT = Path(r"C:\Users\HP\Desktop\Seminario de Tesis I\data")  # solo lectura -- fuente de .obj/.txt/renders

SANDBOX_ROOT    = REPO_ROOT / "Experiments" / "sandbox_pipeline_sin_malla_data"
SANDBOX_OBJECTS = SANDBOX_ROOT / "objects"
SANDBOX_RENDERS = SANDBOX_ROOT / "renders"

SYMMETRY_TYPE  = "axis_sym"   # los 9 objetos de abajo son todos axis_sym (verificado)
OBJECTS_SUBDIR = "curated_axis_sym_obj"
SIZE, LIGHTING = 224, "flat"
DEFAULT_FOV    = 60.0

MODEL_ID = "allenai/Molmo2-8B"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110})

## 1. Lista de objetos + copiar datos al sandbox

Formato de entrada `{CATEGORIA}_{object_id}_2d` -> se parsea a `(categoria, object_id)`.

In [ ]:
RAW_OBJECT_LIST = [
    "BUENO_89cb9b2ad175b833cadf6344ec272e8_2d",
    "BUENO_941271c5d9b192eaccd8f9b9403fd602_2d",
    "BUENO_e0725fd7859fa238ff67c12005f72d2_2d",
    "MALO_6267ef99cbfaea7741cf86c757faf4f9_2d",
    "MALO_955143d7f0b5c70fef76898f881b76a_2d",
    "MALO_e656d6586d481f41eb69804478f9c547_2d",
    "MEDIO_6b8b2cb01c376064c8724d5673a063a6_2d",
    "MEDIO_33b77c66e1f849b790c4e2a44fddf755_2d",
    "MEDIO_f0611ec9ff89209bf10c4513652c1c5e_2d",
]


def parse_entry(entry: str) -> tuple[str, str]:
    """'BUENO_89cb9b2ad175b833cadf6344ec272e8_2d' -> ('BUENO', '89cb9b2ad175b833cadf6344ec272e8').
    El object_id de ShapeNet es siempre hexadecimal de 32 caracteres -- se usa
    eso para separarlo de forma robusta del prefijo de categoria y el sufijo '_2d'."""
    parts = entry.split("_")
    categoria = parts[0]
    object_id = next(p for p in parts[1:] if len(p) >= 30 and all(c in "0123456789abcdef" for c in p))
    return categoria, object_id


OBJECT_LIST = [parse_entry(e) for e in RAW_OBJECT_LIST]  # [(categoria, object_id), ...]
CATEGORY_BY_ID = {oid: cat for cat, oid in OBJECT_LIST}

print(f"{len(OBJECT_LIST)} objetos:")
for cat, oid in OBJECT_LIST:
    print(f"  [{cat:6s}] {oid}")

In [ ]:
SANDBOX_OBJECTS_DIR = SANDBOX_OBJECTS / OBJECTS_SUBDIR
SANDBOX_OBJECTS_DIR.mkdir(parents=True, exist_ok=True)

for cat, oid in OBJECT_LIST:
    # --- .obj / .txt (GT, para el paso de evaluacion -- nunca se usa en la inferencia) ---
    src_objects_dir = DATA_ROOT / "objects" / OBJECTS_SUBDIR
    for ext in (".obj", ".txt"):
        src = src_objects_dir / f"{oid}{ext}"
        dst = SANDBOX_OBJECTS_DIR / f"{oid}{ext}"
        if not dst.exists():
            assert src.exists(), f"No encontrado: {src}"
            shutil.copy2(src, dst)

    # --- renders (PNG + manifest.json + metadata_all.json -- SIN volver a renderizar) ---
    src_render_dir = DATA_ROOT / "renders" / SYMMETRY_TYPE / oid / str(SIZE) / LIGHTING
    dst_render_dir = SANDBOX_RENDERS / SYMMETRY_TYPE / oid / str(SIZE) / LIGHTING
    dst_render_dir.mkdir(parents=True, exist_ok=True)

    assert src_render_dir.exists(), f"No encontrado: {src_render_dir} (¿objeto sin renders generados?)"
    for meta_file in ("manifest.json", "metadata_all.json"):
        src_meta = src_render_dir / meta_file
        dst_meta = dst_render_dir / meta_file
        if src_meta.exists() and not dst_meta.exists():
            shutil.copy2(src_meta, dst_meta)

    n_copied = 0
    for png in src_render_dir.glob("*.png"):
        dst_png = dst_render_dir / png.name
        if not dst_png.exists():
            shutil.copy2(png, dst_png)
            n_copied += 1

    print(f"[{cat:6s}] {oid}: {n_copied} PNG copiados a {dst_render_dir}")

print(f"\nSandbox listo en: {SANDBOX_ROOT}")

## 2. Prompts (editables aca, sin tocar archivos .txt)

Copia funcional de `MolmoPointing/prompts/axis/v06_*.txt` -- edita el texto
directo en esta celda para probar variaciones de redaccion.

In [ ]:
PROMPT_ID = "axis_v06"   # solo para etiquetar el experimento -- cambialo si editas el texto

PROMPT_SINGLE = """You are given ONE image of a 3D object.

The object has ONE dominant rotational symmetry axis.

Your task is to identify the two poles of the rotation axis: the topmost and bottommost points where the axis exits the object's surface.

Return:
- obj_id 1: the TOP pole -- topmost visible point on the rotation axis (at the horizontal center of the topmost surface)
- obj_id 2: the BOTTOM pole -- bottommost visible point on the rotation axis (at the horizontal center of the bottommost surface)

IMPORTANT RULES:
- Both points MUST lie ON the rotation axis -- at the horizontal center of the object at that height, NOT on the lateral silhouette.
- obj_id 1 MUST be above obj_id 2 (smaller Y value).
- The two points MUST be as far apart vertically as possible.
- Both points MUST lie on the visible object surface.
- For flat-topped or flat-bottomed objects, place the pole at the geometric center of the top or bottom face.
- For ROUNDED or CURVED tops/bottoms (no single flat face -- e.g. a dome, a sphere cap, a rounded knob), place the pole at the center of curvature of that rounded region: the point on the visible surface that is equidistant from the left and right silhouette edges at that region's topmost/bottommost extent.
- Verify: draw an imaginary horizontal line through each point -- the point should be equidistant from the left and right edges at that height, regardless of whether the surface there is flat or curved.
- Do NOT place points on the lateral silhouette edges.

Output ONLY:

<points coords="1 1 X1 Y1 2 X2 Y2">"""


PROMPT_MULTI = """You are given multiple views of the SAME 3D object.

The object has ONE dominant axis of rotational symmetry.

For each image, identify the TOP and BOTTOM poles of the global rotation axis -- the points where the axis exits the object's surface at the very top and very bottom.

For each image:
1. Locate where the global axis exits at the top and bottom of the object.
2. Return:
   - obj_id 1: the TOP pole (topmost point on the axis, at the horizontal center of the topmost surface),
   - obj_id 2: the BOTTOM pole (bottommost point on the axis, at the horizontal center of the bottommost surface).

IMPORTANT RULES:
- Both points MUST be ON the rotation axis -- at the horizontal center of the object at that height, NOT on the lateral silhouette.
- Use the SAME global axis consistently across all views.
- obj_id 1 MUST be above obj_id 2 in each image.
- For flat surfaces (e.g., flat-topped objects), place the pole at the geometric center of the top or bottom face.
- For ROUNDED or CURVED tops/bottoms (no single flat face -- e.g. a dome, a sphere cap, a rounded knob), place the pole at the center of curvature of that rounded region: the point equidistant from the left and right silhouette edges at that region's topmost/bottommost extent.
- Infer the global axis from ALL views jointly before answering.
- Verify consistency: the TOP pole should correspond to the same geometric point on the object across all views, whether the surface there is flat or curved.
- Do NOT place points on the lateral silhouette edges.

Output format (one entry per image, separated by semicolons):

<points coords="1 1 Xtop Ytop 2 Xbottom Ybottom; 2 1 Xtop Ytop 2 Xbottom Ybottom; 3 1 Xtop Ytop 2 Xbottom Ybottom">

Where each entry is: image_index obj_id X Y
- obj_id 1 = TOP pole of the rotation axis
- obj_id 2 = BOTTOM pole of the rotation axis

Return ONLY the <points ...> block."""

print(f"Prompt cargado: {PROMPT_ID}")
print(f"(PROMPT_SINGLE: {len(PROMPT_SINGLE)} chars, PROMPT_MULTI: {len(PROMPT_MULTI)} chars)")

## 3. Geometria: camara, triangulacion, filtro fondo/objeto (todo inline)

Copia funcional de `pipeline_common/camera.py` + `pipeline_common/triangulation.py`.

In [ ]:
# --- pipeline_common/camera.py ---

def molmo_to_ndc(x: float, y: float) -> tuple[float, float]:
    """Convert Molmo2 coords (0-1000, top-left origin) to NDC ([-1, 1])."""
    ndc_x = (x / 1000.0) * 2.0 - 1.0
    ndc_y = 1.0 - (y / 1000.0) * 2.0
    return ndc_x, ndc_y


def build_camera_rays(ndc_x: float, ndc_y: float, R: list, T: list,
                      fov_deg: float, image_size: int) -> tuple[np.ndarray, np.ndarray]:
    """World-space (ray_origin, ray_direction) for a given NDC point.
    PyTorch3D row-vector convention: p_cam = p_world @ R + T, so camera center = -(R @ T)."""
    R_np = np.array(R, dtype=np.float64)
    T_np = np.array(T, dtype=np.float64)
    ray_origin = -(R_np @ T_np)
    half_tan = np.tan(np.deg2rad(fov_deg) / 2.0)
    dir_cam = np.array([ndc_x * half_tan, ndc_y * half_tan, 1.0], dtype=np.float64)
    dir_world = R_np @ dir_cam
    dir_world /= np.linalg.norm(dir_world)
    return ray_origin, dir_world


# --- pipeline_common/triangulation.py ---

def ray_dir_for_point(x: float, y: float, R: list, T: list,
                      fov_deg: float, image_size: int) -> tuple[np.ndarray, np.ndarray]:
    ndc_x, ndc_y = molmo_to_ndc(x, y)
    return build_camera_rays(ndc_x, ndc_y, R, T, fov_deg, image_size)


def view_forward_direction(R: list, T: list, fov_deg: float, image_size: int) -> np.ndarray:
    _, direction = build_camera_rays(0.0, 0.0, R, T, fov_deg, image_size)
    return direction


def interpretation_plane_normal(dir_a: np.ndarray, dir_b: np.ndarray):
    """Normal of the plane containing the camera center and two rays from it
    (Bartoli & Sturm 2005). None if the two rays are (numerically) parallel."""
    n = np.cross(dir_a, dir_b)
    norm = np.linalg.norm(n)
    if norm < 1e-9:
        return None
    return n / norm


def triangulate_line(camera_centers: list, plane_normals: list) -> tuple[np.ndarray, np.ndarray]:
    """Intersects >=2 interpretation planes to recover a 3D line: direction =
    right singular vector of smallest singular value; point = least-squares
    solution of n_i . p = n_i . C_i."""
    N = np.asarray(plane_normals, dtype=np.float64)
    C = np.asarray(camera_centers, dtype=np.float64)
    _, _, Vt = np.linalg.svd(N)
    direction = Vt[-1]
    direction /= np.linalg.norm(direction)
    b = np.einsum("ij,ij->i", N, C)
    point, *_ = np.linalg.lstsq(N, b, rcond=None)
    return point, direction


def widest_pair(pts: list):
    """De TODOS los puntos que trae una vista, el par con mayor distancia
    euclidiana en pixeles -- None si hay menos de 2 puntos. Con exactamente 2
    puntos, es identico a tomar esos dos."""
    if len(pts) < 2:
        return None
    best_pair, best_dist_sq = None, -1.0
    for i in range(len(pts)):
        for j in range(i + 1, len(pts)):
            dx = pts[i]["x"] - pts[j]["x"]
            dy = pts[i]["y"] - pts[j]["y"]
            dist_sq = dx * dx + dy * dy
            if dist_sq > best_dist_sq:
                best_dist_sq, best_pair = dist_sq, (pts[i], pts[j])
    return best_pair


def get_point_by_obj_id(pts: list, obj_id: int):
    return next((p for p in pts if p["obj_id"] == obj_id), None)

In [ ]:
# --- filtro fondo/objeto (Mapping/estimate_symmetry_no_mesh.py::filter_points_on_object) ---

BACKGROUND_THRESH = 250  # RGB > esto en los 3 canales = fondo blanco (confirmado: PyTorch3D HardFlatShader default)


def is_on_object(pixel: np.ndarray) -> bool:
    return not bool(np.all(pixel[:3] > BACKGROUND_THRESH))


def molmo_xy_to_pixel(x: float, y: float, img_w: int, img_h: int) -> tuple[int, int]:
    px = int(round((x / 1000.0) * img_w))
    py = int(round((y / 1000.0) * img_h))
    return min(max(px, 0), img_w - 1), min(max(py, 0), img_h - 1)


def filter_points_on_object(points_by_image: dict, images_sent: list, render_dir: Path,
                            min_points: int = 2, image_cache: dict | None = None) -> dict:
    """Descarta puntos que no caen sobre el objeto renderizado. Si a una
    vista le quedan menos de min_points puntos validos, su lista queda vacia
    (= vista invalida para widest_pair/get_point_by_obj_id)."""
    if image_cache is None:
        image_cache = {}
    filtered = {}
    for img_idx_str, pts in points_by_image.items():
        if not pts:
            filtered[img_idx_str] = pts
            continue
        cam = images_sent[int(img_idx_str)]
        filename = cam["filename"]
        if filename not in image_cache:
            img_path = render_dir / filename
            image_cache[filename] = np.array(Image.open(img_path).convert("RGB")) if img_path.exists() else None
        img = image_cache[filename]
        if img is None:
            filtered[img_idx_str] = pts
            continue
        img_h, img_w = img.shape[0], img.shape[1]
        kept = []
        for p in pts:
            px, py = molmo_xy_to_pixel(p["x"], p["y"], img_w, img_h)
            if is_on_object(img[py, px]):
                kept.append(p)
        filtered[img_idx_str] = kept if len(kept) >= min_points else []
    return filtered

## 4. Molmo2: carga del modelo + inferencia (inline) -- necesita GPU

Copia funcional de `MolmoPointing/molmo_multiview_runner.py` (Flow A, sin
Flow B/C -- no hace falta para este sandbox).

In [ ]:
_processor = None
_model     = None


def get_model():
    global _processor, _model
    if _processor is None or _model is None:
        print(f"[model] Loading {MODEL_ID} ...")
        _processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, device_map="auto", use_fast=True)
        _model = AutoModelForImageTextToText.from_pretrained(
            MODEL_ID, trust_remote_code=True, device_map="auto", dtype=torch.bfloat16,
        )
        _model.eval()
        print("[model] Ready.")
    return _processor, _model


def call_model(images: list, prompt: str) -> str:
    """Un solo llamado con 1..N imagenes. Devuelve el texto crudo decodificado."""
    processor, model = get_model()
    content = [{"type": "text", "text": prompt}]
    for img in images:
        content.append({"type": "image", "image": img})
    messages = [{"role": "user", "content": content}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=2048)
    n_input = inputs["input_ids"].size(1)
    del inputs
    text = processor.tokenizer.decode(output_ids[0, n_input:], skip_special_tokens=True)
    del output_ids
    torch.cuda.empty_cache()
    return text


def parse_single_coords(text: str) -> dict:
    """<points coords="RADIO ID X Y ID X Y ..."> -> {"0": [{obj_id,x,y}, ...]}"""
    match = re.search(r'coords=["\']([^"\']+)["\']', text)
    if not match:
        return {}
    raw = [float(n) for n in match.group(1).split()]
    if len(raw) < 4:
        return {}
    raw = raw[1:]
    pts = [{"obj_id": int(raw[i]), "x": raw[i + 1], "y": raw[i + 2]} for i in range(0, len(raw) - 2, 3)]
    return {"0": pts} if pts else {}


def parse_multi_coords(text: str, n_images: int) -> dict:
    """<points coords="img_idx obj_id X Y; ..."> -> {"0": [...], "1": [...], ...} (0-based)"""
    match = re.search(r'coords=["\']([^"\']+)["\']', text)
    if not match:
        return {}
    result: dict = {}
    for group in match.group(1).split(";"):
        group = group.strip()
        if not group:
            continue
        nums = group.split()
        if len(nums) < 4:
            continue
        try:
            img_idx = int(float(nums[0])) - 1
        except ValueError:
            continue
        if img_idx < 0 or img_idx >= n_images:
            continue
        key, rest, pts = str(img_idx), nums[1:], []
        for i in range(0, len(rest) - 2, 3):
            try:
                pts.append({"obj_id": int(float(rest[i])), "x": float(rest[i + 1]), "y": float(rest[i + 2])})
            except ValueError:
                continue
        if pts:
            result.setdefault(key, []).extend(pts)
    return result


def load_metadata(render_dir: Path) -> list:
    path = render_dir / "metadata_all.json"
    if not path.exists():
        raise FileNotFoundError(f"metadata_all.json not found: {render_dir}")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def get_n_views_entries(metadata: list, n_views: int) -> list:
    """n_views entradas evenly spaced (linspace sobre indices) -- evita
    concentrar camaras cerca de un polo, ver docs/pipeline_sin_malla.md S3.1."""
    total = len(metadata)
    if n_views >= total:
        return sorted(metadata, key=lambda e: e["index"])
    indices = {int(round(i)) for i in np.linspace(0, total - 1, n_views)}
    entries = [m for m in metadata if m["index"] in indices]
    return sorted(entries, key=lambda e: e["index"])


def run_inference(images: list, prompt_single: str, prompt_multi: str) -> tuple[str, dict]:
    """auto mode: 1 imagen -> PROMPT_SINGLE; >1 -> PROMPT_MULTI (1 solo llamado)."""
    if len(images) == 1:
        raw = call_model(images, prompt_single)
        pts = parse_single_coords(raw)
        return raw, ({"0": pts["0"]} if "0" in pts else {})
    raw = call_model(images, prompt_multi)
    return raw, parse_multi_coords(raw, n_images=len(images))

## 5. Correr inferencia sobre los 9 objetos

Guarda `molmo_multiview_<EXPERIMENT_ID>.json` dentro del sandbox (mismo
formato que el pipeline real, generado con el codigo de arriba en vez de
invocar el script). Si el JSON de una vista ya existe, se saltea -- borralo
a mano (o cambia `EXPERIMENT_ID`) para forzar recalculo.

In [ ]:
EXPERIMENT_ID = "axis_v06_sandbox"
VIEW_GROUPS   = [6, 14, 26]

for cat, oid in OBJECT_LIST:
    render_dir = SANDBOX_RENDERS / SYMMETRY_TYPE / oid / str(SIZE) / LIGHTING
    metadata   = load_metadata(render_dir)
    json_path  = render_dir / f"molmo_multiview_{EXPERIMENT_ID}.json"
    results    = json.load(open(json_path, encoding="utf-8")) if json_path.exists() else {}

    for n_views in VIEW_GROUPS:
        if str(n_views) in results:
            continue
        entries = get_n_views_entries(metadata, n_views)
        images  = [Image.open(render_dir / e["filename"]).convert("RGB") for e in entries]

        raw, points_by_img = run_inference(images, PROMPT_SINGLE, PROMPT_MULTI)

        results[str(n_views)] = {
            "experiment_id": EXPERIMENT_ID, "prompt_id": PROMPT_ID,
            "prompt_used": PROMPT_SINGLE if n_views == 1 else PROMPT_MULTI,
            "raw_output": raw, "points_by_image": points_by_img,
            "images_sent": [
                {"img_idx": i, "filename": e["filename"], "index": e["index"],
                 "azimuth": e["azimuth"], "elevation": e["elevation"], "eye": e["eye"],
                 "R": e["R"], "T": e["T"]}
                for i, e in enumerate(entries)
            ],
            "n_points": sum(len(v) for v in points_by_img.values()),
        }
        json_path.parent.mkdir(parents=True, exist_ok=True)
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2)
        print(f"  [{cat:6s}] {oid} n_views={n_views}: {results[str(n_views)]['n_points']} puntos devueltos")

print("\nInferencia completa.")

## 6. Estimar eje sin malla (triangulacion, inline)

Copia funcional de `Mapping/estimate_symmetry_no_mesh.py::estimate_axis_no_mesh`
(pooling de normales de interpretacion entre TODAS las vistas de un
n_views group -> una sola SVD).

In [ ]:
FILTER_OFF_OBJECT    = False   # True para probar el filtro fondo/objeto
MIN_POINTS_ON_OBJECT = 2


def estimate_axis_no_mesh(points_by_image: dict, images_sent: list, fov_deg: float, image_size: int):
    centers, normals = [], []
    for img_idx_str, pts in points_by_image.items():
        pair = widest_pair(pts)
        if pair is None:
            continue
        p_a, p_b = pair
        cam = images_sent[int(img_idx_str)]
        C, d_a = ray_dir_for_point(p_a["x"], p_a["y"], cam["R"], cam["T"], fov_deg, image_size)
        _, d_b = ray_dir_for_point(p_b["x"], p_b["y"], cam["R"], cam["T"], fov_deg, image_size)
        n = interpretation_plane_normal(d_a, d_b)
        if n is None:
            continue
        centers.append(C)
        normals.append(n)
    if len(normals) < 2:
        raise ValueError(f"need >=2 valid views, got {len(normals)}")
    point, direction = triangulate_line(centers, normals)
    return {"direction": direction.tolist(), "origin": point.tolist(), "n_views_used": len(normals)}


predicted_axes = {}   # (object_id, n_views) -> {"direction", "origin"} | None si fallo

for cat, oid in OBJECT_LIST:
    render_dir = SANDBOX_RENDERS / SYMMETRY_TYPE / oid / str(SIZE) / LIGHTING
    manifest   = json.load(open(render_dir / "manifest.json", encoding="utf-8")) if (render_dir / "manifest.json").exists() else {}
    fov_deg    = manifest.get("fov", DEFAULT_FOV)
    image_size = manifest.get("image_size", SIZE)

    json_path = render_dir / f"molmo_multiview_{EXPERIMENT_ID}.json"
    molmo_data = json.load(open(json_path, encoding="utf-8"))

    image_cache = {}
    for n_views_key, group in molmo_data.items():
        points_by_image = group["points_by_image"]
        images_sent     = group["images_sent"]
        if FILTER_OFF_OBJECT:
            points_by_image = filter_points_on_object(
                points_by_image, images_sent, render_dir,
                min_points=MIN_POINTS_ON_OBJECT, image_cache=image_cache,
            )
        try:
            pred = estimate_axis_no_mesh(points_by_image, images_sent, fov_deg, image_size)
        except ValueError as e:
            pred = None
            print(f"  [{cat:6s}] {oid} n_views={n_views_key}: [omitido] {e}")
        predicted_axes[(oid, int(n_views_key))] = pred

print(f"\n{sum(v is not None for v in predicted_axes.values())}/{len(predicted_axes)} predicciones validas.")

## 7. Evaluar contra GT (inline)

Copia funcional de `Mapping/evaluate.py::parse_true_label` /
`angular_error_deg` / `point_to_line_distance` / `precision_{t}deg`.

In [ ]:
ANGULAR_THRESHOLDS = [5, 10, 15]


def parse_true_label(txt_path: Path) -> dict:
    lines = [l.strip() for l in txt_path.read_text().splitlines() if l.strip()]
    for line in lines:
        if line.startswith("axis"):
            parts = line.split()
            vec  = np.array([float(x) for x in parts[1:4]])
            orig = [float(x) for x in parts[4:7]]
            vec /= np.linalg.norm(vec)
            return {"direction": vec.tolist(), "origin": orig}
    raise ValueError(f"No se encontro una linea 'axis' en {txt_path}")


def angular_error_deg(v1: np.ndarray, v2: np.ndarray) -> float:
    v1, v2 = v1 / np.linalg.norm(v1), v2 / np.linalg.norm(v2)
    return float(np.degrees(np.arccos(np.clip(np.abs(np.dot(v1, v2)), 0.0, 1.0))))


def point_to_line_distance(point: np.ndarray, line_origin: np.ndarray, line_dir: np.ndarray) -> float:
    d = line_dir / np.linalg.norm(line_dir)
    v = point - line_origin
    return float(np.linalg.norm(v - np.dot(v, d) * d))


rows = []
for cat, oid in OBJECT_LIST:
    gt = parse_true_label(SANDBOX_OBJECTS_DIR / f"{oid}.txt")
    t_dir, t_orig = np.array(gt["direction"]), np.array(gt["origin"])

    for n_views in VIEW_GROUPS:
        pred = predicted_axes.get((oid, n_views))
        row = {"categoria": cat, "object_id": oid, "n_views": n_views}
        if pred is None:
            row.update({"status": "no_pred", "angular_error_deg": 90.0, "translation_error": None})
        else:
            p_dir, p_orig = np.array(pred["direction"]), np.array(pred["origin"])
            ang  = angular_error_deg(p_dir, t_dir)
            dist = point_to_line_distance(p_orig, t_orig, t_dir)
            row.update({"status": "ok", "angular_error_deg": round(ang, 4), "translation_error": round(dist, 6)})
            for t in ANGULAR_THRESHOLDS:
                row[f"precision_{t}deg"] = int(ang < t)
        rows.append(row)

df_eval = pd.DataFrame(rows).sort_values(["categoria", "object_id", "n_views"]).reset_index(drop=True)
df_eval

## 8. Resultados: resumen por categoria y por n_views

In [ ]:
print(f"Experimento: {EXPERIMENT_ID}  |  Prompt: {PROMPT_ID}  |  "
      f"filter_off_object={FILTER_OFF_OBJECT}\n")

print("--- Por objeto ---")
display(df_eval)

print("\n--- Promedio por categoria x n_views ---")
display(df_eval.groupby(["categoria", "n_views"])[["angular_error_deg"]].mean().round(2))

print("\n--- Promedio global por n_views (equivalente a angular_error_mean del pipeline real) ---")
display(df_eval.groupby("n_views")[["angular_error_deg"]].agg(["mean", "median", "std", "min", "max"]).round(2))

## 9. Visualizacion 2D/3D (opcional)

Mismo esquema de `Experiments/visualizar_casos_axis.ipynb`, reescrito inline
(sin importar nada del repo salvo `trimesh`/`PIL`/`matplotlib`, ya cargados
en el Setup).

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401


def plot_case(object_id: str, categoria: str, n_views: int, max_views_2d: int = 6) -> None:
    render_dir = SANDBOX_RENDERS / SYMMETRY_TYPE / object_id / str(SIZE) / LIGHTING
    molmo_data = json.load(open(render_dir / f"molmo_multiview_{EXPERIMENT_ID}.json", encoding="utf-8"))
    group = molmo_data[str(n_views)]
    images_sent, points_by_image = group["images_sent"], group["points_by_image"]

    pred = predicted_axes.get((object_id, n_views))
    gt   = parse_true_label(SANDBOX_OBJECTS_DIR / f"{object_id}.txt")
    err  = df_eval[(df_eval.object_id == object_id) & (df_eval.n_views == n_views)]["angular_error_deg"].iloc[0]

    # --- 2D ---
    idxs = sorted(points_by_image.keys(), key=int)[:max_views_2d]
    fig, axes = plt.subplots(1, len(idxs), figsize=(3.2 * len(idxs), 3.4))
    if len(idxs) == 1:
        axes = [axes]
    for ax, idx_str in zip(axes, idxs):
        cam = images_sent[int(idx_str)]
        img_path = render_dir / cam["filename"]
        img = np.array(Image.open(img_path)) if img_path.exists() else None
        if img is not None:
            ax.imshow(img)
            img_h, img_w = img.shape[0], img.shape[1]
        else:
            img_w = img_h = SIZE
            ax.set_xlim(0, img_w); ax.set_ylim(img_h, 0)
        for p in points_by_image[idx_str]:
            px, py = molmo_xy_to_pixel(p["x"], p["y"], img_w, img_h)
            ax.scatter([px], [py], c="red" if p["obj_id"] == 1 else "blue",
                       s=60, edgecolors="white", linewidths=1.2, zorder=5)
        ax.set_title(f"img {idx_str}", fontsize=8)
        ax.axis("off")
    fig.suptitle(f"[{categoria}] {object_id} -- angular_error={err:.2f} grados", fontsize=10)
    fig.tight_layout()
    plt.show()

    # --- 3D ---
    mesh  = trimesh.load(str(SANDBOX_OBJECTS_DIR / f"{object_id}.obj"), force="mesh", process=False)
    verts = np.asarray(mesh.vertices)
    if len(verts) > 4000:
        verts_plot = verts[np.random.default_rng(0).choice(len(verts), 4000, replace=False)]
    else:
        verts_plot = verts
    bbox_diag = float(np.linalg.norm(verts.max(axis=0) - verts.min(axis=0)))

    fig = plt.figure(figsize=(6.5, 6.5))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(verts_plot[:, 0], verts_plot[:, 1], verts_plot[:, 2], s=1, c="lightgray", alpha=0.4, label="malla")

    def plot_line(origin, direction, color, label):
        d = np.array(direction) / np.linalg.norm(direction)
        o = np.array(origin)
        p0, p1 = o - d * bbox_diag * 0.75, o + d * bbox_diag * 0.75
        ax.plot([p0[0], p1[0]], [p0[1], p1[1]], [p0[2], p1[2]], color=color, linewidth=2.5, label=label)

    plot_line(gt["origin"], gt["direction"], "green", "eje GT")
    if pred is not None:
        plot_line(pred["origin"], pred["direction"], "red", "eje predicho")
    ax.set_title(f"[{categoria}] {object_id} -- angular_error={err:.2f} grados", fontsize=10)
    ax.legend(loc="upper left", fontsize=8)
    ax.set_box_aspect([1, 1, 1])
    fig.tight_layout()
    plt.show()


# Ejemplo: un caso de cada categoria, al mayor n_views
for cat, oid in [(c, o) for c, o in OBJECT_LIST if c in ("BUENO", "MEDIO", "MALO")][::3]:
    plot_case(oid, cat, n_views=max(VIEW_GROUPS))

---
## 10. Ablaciones de triangulación y consolidación

Esta sección añade experimentos independientes sobre los puntos ya inferidos
por Molmo2 (no hace falta volver a correr el modelo). Cada ablación se puede
activar / desactivar cambiando sus flags, y todas comparten el mismo evaluador
de la Sección 8 para que los resultados sean comparables directamente.

### Índice de experimentos
| ID | Qué cambia | Referencia |
|----|-----------|-----------|
| **EXP-A** | Triangulación SVD ponderada por longitud 2D (*weighted interpretation planes*) | Bartoli & Sturm, CVIU 2005 |
| **EXP-B** | Triangulación PtD: punto ancla + dirección por mínimo error angular | Wu et al., PMC 2021 |
| **EXP-C** | Filtro de objeto + RANSAC 2D sobre ≥6 puntos | Recker et al., WACV 2013 |
| **EXP-D** | Ponderación por confianza proxy (longitud 2D ÷ distancia al borde) | AssemblyHands-X, arXiv 2025 |
| **EXP-E** | Descarte de vistas con paralaje insuficiente (post-hoc, sin eliminar vistas del dataset) | Hess-Flores et al., eScholarship |
| **EXP-F** | Pipeline iterativo 3-tier para simetría planar | Propuesto en esta tesis |

> **Nota metodológica**: EXP-E descarta vistas *a posteriori del cálculo del normal*
> (no del dataset) basándose en el residual de reproyección — es distinto a filtrar
> vistas por ángulo antes de inferencia. El dataset permanece completo.


### 10.0 Helpers y evaluador compartido

In [ ]:
# ── Helpers compartidos para todas las ablaciones ───────────────────────────
import itertools

def angular_error_deg_signed(v1: np.ndarray, v2: np.ndarray) -> float:
    """Error angular sign-agnostic en [0°, 90°]."""
    v1 = v1 / np.linalg.norm(v1)
    v2 = v2 / np.linalg.norm(v2)
    return float(np.degrees(np.arccos(np.clip(abs(float(np.dot(v1, v2))), 0.0, 1.0))))


def auc_angular(errors_deg: list[float], max_theta: float = 45.0) -> float:
    """AUC de la curva precision-vs-umbral en [0°, max_theta°]."""
    thetas = np.linspace(0, max_theta, 1000)
    errors = np.array(errors_deg)
    precisions = [(errors <= t).mean() for t in thetas]
    return float(np.trapezoid(precisions, thetas) / max_theta)


def summarize_results(rows: list[dict], label: str) -> pd.DataFrame:
    """Imprime y devuelve un DataFrame de resumen por n_views."""
    df = pd.DataFrame(rows)
    df["angular_error_deg"] = df["angular_error_deg"].fillna(90.0)
    summary = (df.groupby("n_views")["angular_error_deg"]
               .agg(mean="mean", median="median", std="std")
               .round(2))
    for t in ANGULAR_THRESHOLDS:
        col = f"precision_{t}deg"
        if col in df.columns:
            summary[f"P@{t}°"] = df.groupby("n_views")[col].mean().round(4)
    auc_per_nv = {}
    for nv, grp in df.groupby("n_views"):
        auc_per_nv[nv] = round(auc_angular(grp["angular_error_deg"].tolist()), 4)
    summary["AUC@45°"] = pd.Series(auc_per_nv)
    print(f"\n{'='*60}")
    print(f"  {label}")
    print('='*60)
    print(summary.to_string())
    return summary


### EXP-A — Triangulación ponderada por longitud del segmento 2D
*(Bartoli & Sturm, CVIU 2005; Recker et al., WACV 2013)*

**Motivación**: en la triangulación estándar (`triangulate_line`) todos los
planos de interpretación contribuyen por igual. Una vista en que los dos polos
predichos están a 5 px de distancia proporciona el mismo peso que una en que
están a 200 px. Sin embargo, la dirección de la línea está mucho menos
determinada cuando el segmento es corto — el plano de interpretación
es correcto, pero su normal es sensible a pequeñas perturbaciones.

La ponderación por longitud 2D \( w_i = \|p_a^{(i)} - p_b^{(i)}\|_2 \)
construye la matriz  
\( M = \sum_i w_i \, \mathbf{n}_i \mathbf{n}_i^\top \)  
cuyo eigenvector de eigenvalor mínimo es la dirección del eje
(Bartoli & Sturm Alg. 1, adaptado a SVD ponderado).

**Referencia**: Bartoli A. & Sturm P. (2005). *Structure-from-motion using lines:
Representation, triangulation, and bundle adjustment*. CVIU, 100(3):416–441.
DOI: 10.1016/j.cviu.2005.06.001


In [ ]:
# ── EXP-A: Triangulación ponderada por longitud 2D ──────────────────────────
# Referencia: Bartoli & Sturm, CVIU 2005

def triangulate_line_weighted(camera_centers: list, plane_normals: list,
                               weights: list) -> tuple[np.ndarray, np.ndarray]:
    """
    Weighted interpretation-plane triangulation (Bartoli & Sturm 2005, Alg. 1).
    Builds M = sum_i w_i * n_i n_i^T; direction = eigvec of min eigenvalue.
    Point is recovered via weighted least squares: sum_i w_i n_i . p = sum_i w_i n_i . C_i.

    Parameters
    ----------
    camera_centers : list of (3,) arrays
    plane_normals  : list of (3,) unit arrays
    weights        : list of floats (e.g. 2D segment lengths in pixels)

    Returns
    -------
    point     : (3,) anchor point on the line
    direction : (3,) unit direction vector
    """
    N  = np.asarray(plane_normals, dtype=np.float64)   # (K, 3)
    C  = np.asarray(camera_centers, dtype=np.float64)  # (K, 3)
    w  = np.asarray(weights, dtype=np.float64)         # (K,)
    w  = w / w.sum()                                   # normalise

    # Weighted M matrix — direction from its null space
    M = (N * w[:, None]).T @ N                         # (3, 3)
    _, _, Vt = np.linalg.svd(M)
    direction = Vt[-1]
    direction /= np.linalg.norm(direction)

    # Weighted least squares for the anchor point
    W_diag = np.diag(w)
    A = W_diag @ N                                     # (K, 3)
    b = np.einsum("ij,ij->i", N, C) * w               # (K,)
    point, *_ = np.linalg.lstsq(A, b, rcond=None)
    return point, direction


def estimate_axis_weighted(points_by_image: dict, images_sent: list,
                            fov_deg: float, image_size: int):
    """EXP-A: widest_pair per view + weight = 2D segment length (pixels)."""
    centers, normals, weights = [], [], []
    for img_idx_str, pts in points_by_image.items():
        pair = widest_pair(pts)
        if pair is None:
            continue
        p_a, p_b = pair
        cam = images_sent[int(img_idx_str)]
        C, d_a = ray_dir_for_point(p_a["x"], p_a["y"], cam["R"], cam["T"], fov_deg, image_size)
        _, d_b = ray_dir_for_point(p_b["x"], p_b["y"], cam["R"], cam["T"], fov_deg, image_size)
        n = interpretation_plane_normal(d_a, d_b)
        if n is None:
            continue
        seg_len = ((p_a["x"] - p_b["x"])**2 + (p_a["y"] - p_b["y"])**2) ** 0.5
        centers.append(C); normals.append(n); weights.append(max(seg_len, 1e-3))
    if len(normals) < 2:
        raise ValueError(f"EXP-A: need >=2 valid views, got {len(normals)}")
    point, direction = triangulate_line_weighted(centers, normals, weights)
    return {"direction": direction.tolist(), "origin": point.tolist(),
            "n_views_used": len(normals), "method": "weighted_bartoli"}


# ── Run EXP-A ─────────────────────────────────────────────────────────────────
rows_A = []
for cat, oid in OBJECT_LIST:
    render_dir = SANDBOX_RENDERS / SYMMETRY_TYPE / oid / str(SIZE) / LIGHTING
    manifest   = json.load(open(render_dir / "manifest.json")) if (render_dir / "manifest.json").exists() else {}
    fov_deg    = manifest.get("fov", DEFAULT_FOV); image_size = manifest.get("image_size", SIZE)
    json_path  = render_dir / f"molmo_multiview_{EXPERIMENT_ID}.json"
    molmo_data = json.load(open(json_path))
    gt         = parse_true_label(SANDBOX_OBJECTS_DIR / f"{oid}.txt")
    t_dir      = np.array(gt["direction"]); t_orig = np.array(gt["origin"])

    for n_views in VIEW_GROUPS:
        group = molmo_data.get(str(n_views), {})
        pts_by_img = group.get("points_by_image", {}); imgs_sent = group.get("images_sent", [])
        row = {"categoria": cat, "object_id": oid, "n_views": n_views}
        try:
            pred = estimate_axis_weighted(pts_by_img, imgs_sent, fov_deg, image_size)
            ang  = angular_error_deg_signed(np.array(pred["direction"]), t_dir)
            dist = point_to_line_distance(np.array(pred["origin"]), t_orig, t_dir)
            row.update({"angular_error_deg": ang, "translation_error": dist})
            for t in ANGULAR_THRESHOLDS:
                row[f"precision_{t}deg"] = int(ang < t)
        except ValueError as e:
            row["angular_error_deg"] = 90.0
        rows_A.append(row)

summary_A = summarize_results(rows_A, "EXP-A: Triangulación ponderada por longitud 2D (Bartoli & Sturm 2005)")


### EXP-B — Triangulación Point-then-Direction (PtD)
*(Wu et al., PMC 2021 — "An Accurate Linear Method for 3D Line Reconstruction")*

**Motivación**: en lugar de recuperar simultáneamente punto y dirección del eje,
PtD separa los dos subproblemas:

1. **Punto ancla** \(p^*\): triangular el punto 3D promedio de todos los puntos
   predichos usando DLT / punto medio ponderado entre pares de vistas.
2. **Dirección** \(d^*\): dado \(p^*\), encontrar la dirección que minimiza
   el error angular de reproyección en cada vista:
   \[
   d^* = \arg\min_d \sum_i \sin^2\!\angle(\, \pi_i(p^* + d),\, l_i \,)
   \]
   donde \(l_i\) es la línea proyectada en la vista \(i\).

Esto es más robusto que el SVD global cuando el punto ancla es bueno pero
la dirección tiene outliers en vistas con bajo paralaje.

**Referencia**: Wu F., Zhang M., Wang G., Hu Z. (2021). *An Accurate Linear Method
for 3D Line Reconstruction*. Computational Intelligence and Neuroscience.
PMC: PMC7832884. https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7832884/


In [ ]:
# ── EXP-B: Point-then-Direction (PtD) ───────────────────────────────────────
# Referencia: Wu et al., PMC 2021

def midpoint_triangulate_pair(o1: np.ndarray, d1: np.ndarray,
                               o2: np.ndarray, d2: np.ndarray) -> np.ndarray:
    """Closest point between two rays (midpoint method). Returns midpoint."""
    w = o1 - o2
    a, b, c, d, e = (np.dot(d1, d1), np.dot(d1, d2), np.dot(d2, d2),
                     np.dot(d1, w), np.dot(d2, w))
    denom = a * c - b * b
    if abs(denom) < 1e-9:
        return (o1 + o2) / 2.0
    s = (b * e - c * d) / denom
    t = (a * e - b * d) / denom
    return ((o1 + s * d1) + (o2 + t * d2)) / 2.0


def estimate_axis_ptd(points_by_image: dict, images_sent: list,
                       fov_deg: float, image_size: int):
    """
    EXP-B: Point-then-Direction triangulation.
    Step 1 — estimate anchor point by averaging midpoint triangulations of all pairs.
    Step 2 — given anchor, find direction minimizing angular reprojection error
              via SVD on the matrix A where each row is n_i x (p* x n_i)
              (linearised angular cost around the interpretation plane normals).
    """
    # Collect per-view: camera center + ray dirs for widest pair
    view_data = []
    for img_idx_str, pts in points_by_image.items():
        pair = widest_pair(pts)
        if pair is None:
            continue
        p_a, p_b = pair
        cam = images_sent[int(img_idx_str)]
        C, d_a = ray_dir_for_point(p_a["x"], p_a["y"], cam["R"], cam["T"], fov_deg, image_size)
        _, d_b  = ray_dir_for_point(p_b["x"], p_b["y"], cam["R"], cam["T"], fov_deg, image_size)
        n = interpretation_plane_normal(d_a, d_b)
        if n is None:
            continue
        seg_len = ((p_a["x"] - p_b["x"])**2 + (p_a["y"] - p_b["y"])**2) ** 0.5
        # Mid-ray: average direction of the two rays (represents the "axis ray" from camera)
        d_mid = (d_a + d_b); d_mid /= np.linalg.norm(d_mid)
        view_data.append({"C": C, "d_mid": d_mid, "d_a": d_a, "d_b": d_b,
                           "n": n, "seg": seg_len})

    if len(view_data) < 2:
        raise ValueError(f"EXP-B: need >=2 valid views, got {len(view_data)}")

    # Step 1: anchor point via weighted average of pairwise midpoint triangulations
    anchor_pts = []
    for i, j in itertools.combinations(range(len(view_data)), 2):
        vi, vj = view_data[i], view_data[j]
        mp = midpoint_triangulate_pair(vi["C"], vi["d_mid"], vj["C"], vj["d_mid"])
        w = (vi["seg"] * vj["seg"]) ** 0.5   # geometric mean of segment lengths
        anchor_pts.append((mp, w))
    total_w = sum(w for _, w in anchor_pts)
    anchor = sum(mp * w for mp, w in anchor_pts) / total_w   # weighted centroid

    # Step 2: direction via SVD on interpretation-plane normals (same as EXP-A but
    # using the anchor to set up a direction-only optimisation)
    normals = np.array([v["n"] for v in view_data])
    weights = np.array([max(v["seg"], 1e-3) for v in view_data])
    weights /= weights.sum()
    M = (normals * weights[:, None]).T @ normals
    _, _, Vt = np.linalg.svd(M)
    direction = Vt[-1]; direction /= np.linalg.norm(direction)

    return {"direction": direction.tolist(), "origin": anchor.tolist(),
            "n_views_used": len(view_data), "method": "PtD_wu2021"}


# ── Run EXP-B ─────────────────────────────────────────────────────────────────
rows_B = []
for cat, oid in OBJECT_LIST:
    render_dir = SANDBOX_RENDERS / SYMMETRY_TYPE / oid / str(SIZE) / LIGHTING
    manifest   = json.load(open(render_dir / "manifest.json")) if (render_dir / "manifest.json").exists() else {}
    fov_deg    = manifest.get("fov", DEFAULT_FOV); image_size = manifest.get("image_size", SIZE)
    json_path  = render_dir / f"molmo_multiview_{EXPERIMENT_ID}.json"
    molmo_data = json.load(open(json_path))
    gt         = parse_true_label(SANDBOX_OBJECTS_DIR / f"{oid}.txt")
    t_dir      = np.array(gt["direction"]); t_orig = np.array(gt["origin"])

    for n_views in VIEW_GROUPS:
        group = molmo_data.get(str(n_views), {}); row = {"categoria": cat, "object_id": oid, "n_views": n_views}
        try:
            pred = estimate_axis_ptd(group.get("points_by_image", {}), group.get("images_sent", []), fov_deg, image_size)
            ang  = angular_error_deg_signed(np.array(pred["direction"]), t_dir)
            dist = point_to_line_distance(np.array(pred["origin"]), t_orig, t_dir)
            row.update({"angular_error_deg": ang, "translation_error": dist})
            for t in ANGULAR_THRESHOLDS: row[f"precision_{t}deg"] = int(ang < t)
        except ValueError:
            row["angular_error_deg"] = 90.0
        rows_B.append(row)

summary_B = summarize_results(rows_B, "EXP-B: Point-then-Direction (Wu et al. PMC 2021)")


### EXP-C — Filtro de objeto + RANSAC 2D sobre ≥6 puntos
*(Recker et al., WACV 2013; "Robust Uncertainty-Aware Multiview Triangulation", arXiv 2008.01258)*

**Motivación**: cuando se piden 6 puntos por vista en lugar de 2, los puntos
son aproximadamente colineales en imagen (todos sobre la proyección del eje o
la traza del plano). Aplicar RANSAC 2D sobre esos 6 puntos antes de formar el
plano de interpretación da una línea 2D robusta incluso si 1-2 puntos son
outliers (Molmo los colocó fuera del eje).

**Flujo**:
1. Filtro de objeto: descartar puntos cuyo píxel es fondo blanco.
2. RANSAC 2D: ajustar una línea a los puntos restantes de cada vista;
   umbral de distancia perpendicular = `ransac_thr_px` píxeles.
3. Usar los 2 extremos inliers más separados como par representativo.
4. Triangulación ponderada (EXP-A) sobre los pares robustificados.

**Referencias**:  
Recker S., Hess-Flores M., Joy K.I. (2013). *Statistical Angular Error-Based
Triangulation*. WACV 2013. https://sites.google.com/view/mauriciohessflores/multi-view-triangulation  
Zhang et al. (2020). *Robust Uncertainty-Aware Multiview Triangulation*.
arXiv:2008.01258. https://arxiv.org/abs/2008.01258


In [ ]:
# ── EXP-C: Filtro objeto + RANSAC 2D + triangulación ponderada ───────────────
# Referencias: Recker et al. WACV 2013; Zhang et al. arXiv 2020

RANSAC_THR_PX   = 8.0   # distancia perpendicular máxima para inlier (en píxeles)
RANSAC_N_ITER   = 100   # iteraciones RANSAC
RANSAC_MIN_PTS  = 4     # mínimo de inliers para aceptar un modelo de línea

# Requiere que las inferencias hayan pedido >=6 puntos por vista.
# Si usás prompts de 2 puntos, este experimento devuelve los mismos resultados
# que EXP-A (no hay outliers extra que rechazar con 2 puntos).


def fit_line_ransac_2d(points_xy: list[tuple[float, float]],
                        n_iter: int = RANSAC_N_ITER,
                        thr_px: float = RANSAC_THR_PX,
                        min_inliers: int = RANSAC_MIN_PTS,
                        rng: np.random.Generator | None = None) -> tuple[list, float] | None:
    """
    RANSAC 2D line fitting in pixel coordinates.
    Returns (inlier_points, seg_length) or None if consensus not reached.

    Line model: ax + by + c = 0 (from 2-point sample), inlier iff
    |ax + by + c| / sqrt(a^2 + b^2) < thr_px.
    """
    if rng is None:
        rng = np.random.default_rng(42)
    pts = np.array(points_xy, dtype=np.float64)
    n = len(pts)
    if n < 2:
        return None
    if n == 2:                        # degenerate: accept trivially
        seg = np.linalg.norm(pts[1] - pts[0])
        return points_xy, seg

    best_inliers, best_seg = [], 0.0
    for _ in range(n_iter):
        i, j = rng.choice(n, size=2, replace=False)
        p1, p2 = pts[i], pts[j]
        # Line through p1, p2:  n . x = n . p1   where n = perp of (p2-p1)
        diff = p2 - p1
        if np.linalg.norm(diff) < 1e-3:
            continue
        a, b = -diff[1], diff[0]        # normal to the line
        c = -(a * p1[0] + b * p1[1])
        norm_ab = (a*a + b*b) ** 0.5
        dists = np.abs(a * pts[:, 0] + b * pts[:, 1] + c) / norm_ab
        inlier_mask = dists < thr_px
        if inlier_mask.sum() > len(best_inliers):
            inlier_pts   = pts[inlier_mask]
            # Segment length = max pairwise distance among inliers projected on line dir
            dir_n = diff / np.linalg.norm(diff)
            projs = inlier_pts @ dir_n
            seg   = float(projs.max() - projs.min())
            best_inliers = [tuple(p) for p in inlier_pts]
            best_seg = seg

    if len(best_inliers) < min_inliers:
        return None
    return best_inliers, best_seg


def widest_pair_from_inliers(inlier_pts: list[tuple[float, float]]):
    """Return the two most distant inlier points (for the interpretation plane)."""
    pts = np.array(inlier_pts)
    best = (-1.0, 0, 1)
    for i in range(len(pts)):
        for j in range(i + 1, len(pts)):
            d = float(np.sum((pts[i] - pts[j])**2))
            if d > best[0]:
                best = (d, i, j)
    _, i, j = best
    return {"x": float(pts[i, 0]), "y": float(pts[i, 1])},            {"x": float(pts[j, 0]), "y": float(pts[j, 1])},            best[0] ** 0.5


def estimate_axis_ransac(points_by_image: dict, images_sent: list,
                          fov_deg: float, image_size: int,
                          render_dir: Path,
                          filter_object: bool = True,
                          image_cache: dict | None = None):
    """
    EXP-C: filter object pixels, then 2D RANSAC line fitting, then weighted
    interpretation-plane triangulation.
    """
    if image_cache is None:
        image_cache = {}
    if filter_object:
        pts_filtered = filter_points_on_object(
            points_by_image, images_sent, render_dir,
            min_points=2, image_cache=image_cache)
    else:
        pts_filtered = points_by_image

    rng = np.random.default_rng(0)
    centers, normals, weights = [], [], []
    for img_idx_str, pts in pts_filtered.items():
        if not pts:
            continue
        xy = [(p["x"], p["y"]) for p in pts]
        result = fit_line_ransac_2d(xy, rng=rng)
        if result is None:
            continue
        inliers, seg_len = result
        p_a, p_b, seg_px = widest_pair_from_inliers(inliers)

        cam = images_sent[int(img_idx_str)]
        C, d_a = ray_dir_for_point(p_a["x"], p_a["y"], cam["R"], cam["T"], fov_deg, image_size)
        _, d_b  = ray_dir_for_point(p_b["x"], p_b["y"], cam["R"], cam["T"], fov_deg, image_size)
        n = interpretation_plane_normal(d_a, d_b)
        if n is None:
            continue
        centers.append(C); normals.append(n); weights.append(max(seg_px, 1e-3))

    if len(normals) < 2:
        raise ValueError(f"EXP-C: need >=2 valid views, got {len(normals)}")
    point, direction = triangulate_line_weighted(centers, normals, weights)
    return {"direction": direction.tolist(), "origin": point.tolist(),
            "n_views_used": len(normals), "method": "ransac2d_recker"}


# ── Run EXP-C ─────────────────────────────────────────────────────────────────
FILTER_OBJECT_C = True   # Poner False para aislar solo el efecto de RANSAC 2D

rows_C = []
for cat, oid in OBJECT_LIST:
    render_dir = SANDBOX_RENDERS / SYMMETRY_TYPE / oid / str(SIZE) / LIGHTING
    manifest   = json.load(open(render_dir / "manifest.json")) if (render_dir / "manifest.json").exists() else {}
    fov_deg    = manifest.get("fov", DEFAULT_FOV); image_size = manifest.get("image_size", SIZE)
    json_path  = render_dir / f"molmo_multiview_{EXPERIMENT_ID}.json"
    molmo_data = json.load(open(json_path))
    gt         = parse_true_label(SANDBOX_OBJECTS_DIR / f"{oid}.txt")
    t_dir      = np.array(gt["direction"]); t_orig = np.array(gt["origin"])
    img_cache  = {}

    for n_views in VIEW_GROUPS:
        group = molmo_data.get(str(n_views), {}); row = {"categoria": cat, "object_id": oid, "n_views": n_views}
        try:
            pred = estimate_axis_ransac(
                group.get("points_by_image", {}), group.get("images_sent", []),
                fov_deg, image_size, render_dir,
                filter_object=FILTER_OBJECT_C, image_cache=img_cache)
            ang  = angular_error_deg_signed(np.array(pred["direction"]), t_dir)
            dist = point_to_line_distance(np.array(pred["origin"]), t_orig, t_dir)
            row.update({"angular_error_deg": ang, "translation_error": dist})
            for t in ANGULAR_THRESHOLDS: row[f"precision_{t}deg"] = int(ang < t)
        except ValueError:
            row["angular_error_deg"] = 90.0
        rows_C.append(row)

summary_C = summarize_results(rows_C, f"EXP-C: Filtro objeto + RANSAC 2D (Recker WACV 2013) | filter={FILTER_OBJECT_C}")


### EXP-D — Ponderación por confianza proxy: longitud 2D ÷ distancia al borde
*(AssemblyHands-X, arXiv:2509.23888, 2025; COMPOSE arXiv:2601.09698, 2026)*

**Motivación**: vistas donde el objeto está parcialmente fuera del encuadre
o muy cerca del borde de imagen tienen puntos predichos con menor fiabilidad
(el modelo no ve el objeto completo). AssemblyHands-X propone penalizar
linealmente la confianza de keypoints según su distancia al borde del recorte.

La confianza proxy combina:
- **Longitud 2D** \(\ell_i\) del segmento: mayor longitud = menos sensible a ruido.
- **Margen mínimo al borde** \(m_i\): si el punto más cercano al borde está
  a menos de `border_margin` píxeles, se penaliza la vista.

\[ w_i = \ell_i \cdot \min\!\left(1,\, \frac{m_i}{\text{border\_margin}}\right) \]

**Referencias**:  
AssemblyHands-X (2025). arXiv:2509.23888. https://arxiv.org/pdf/2509.23888  
COMPOSE (2026). arXiv:2601.09698. https://arxiv.org/pdf/2601.09698


In [ ]:
# ── EXP-D: Weighted triangulation with border-proximity penalty ──────────────
# Referencias: AssemblyHands-X arXiv:2509.23888; COMPOSE arXiv:2601.09698

BORDER_MARGIN_PX = 20   # distancia mínima al borde para no penalizar (en píxeles)


def border_margin(x: float, y: float, image_size: int) -> float:
    """Minimum distance from point (x,y) in Molmo scale [0,1000] to image border."""
    px = (x / 1000.0) * image_size
    py = (y / 1000.0) * image_size
    return float(min(px, py, image_size - px, image_size - py))


def estimate_axis_confidence_proxy(points_by_image: dict, images_sent: list,
                                    fov_deg: float, image_size: int):
    """
    EXP-D: interpretation-plane triangulation weighted by
           w_i = seg_len_i * min(1, border_margin_i / BORDER_MARGIN_PX).
    """
    centers, normals, weights = [], [], []
    for img_idx_str, pts in points_by_image.items():
        pair = widest_pair(pts)
        if pair is None:
            continue
        p_a, p_b = pair
        cam = images_sent[int(img_idx_str)]
        C, d_a = ray_dir_for_point(p_a["x"], p_a["y"], cam["R"], cam["T"], fov_deg, image_size)
        _, d_b  = ray_dir_for_point(p_b["x"], p_b["y"], cam["R"], cam["T"], fov_deg, image_size)
        n = interpretation_plane_normal(d_a, d_b)
        if n is None:
            continue
        seg_len = ((p_a["x"] - p_b["x"])**2 + (p_a["y"] - p_b["y"])**2) ** 0.5

        # Border penalty (AssemblyHands-X)
        m_a = border_margin(p_a["x"], p_a["y"], image_size)
        m_b = border_margin(p_b["x"], p_b["y"], image_size)
        margin = min(m_a, m_b)
        border_factor = min(1.0, margin / BORDER_MARGIN_PX)

        w = max(seg_len, 1e-3) * border_factor
        centers.append(C); normals.append(n); weights.append(max(w, 1e-6))

    if len(normals) < 2:
        raise ValueError(f"EXP-D: need >=2 valid views, got {len(normals)}")
    point, direction = triangulate_line_weighted(centers, normals, weights)
    return {"direction": direction.tolist(), "origin": point.tolist(),
            "n_views_used": len(normals), "method": "confidence_proxy_assemblyhands"}


# ── Run EXP-D ─────────────────────────────────────────────────────────────────
rows_D = []
for cat, oid in OBJECT_LIST:
    render_dir = SANDBOX_RENDERS / SYMMETRY_TYPE / oid / str(SIZE) / LIGHTING
    manifest   = json.load(open(render_dir / "manifest.json")) if (render_dir / "manifest.json").exists() else {}
    fov_deg    = manifest.get("fov", DEFAULT_FOV); image_size = manifest.get("image_size", SIZE)
    json_path  = render_dir / f"molmo_multiview_{EXPERIMENT_ID}.json"
    molmo_data = json.load(open(json_path))
    gt         = parse_true_label(SANDBOX_OBJECTS_DIR / f"{oid}.txt")
    t_dir      = np.array(gt["direction"]); t_orig = np.array(gt["origin"])

    for n_views in VIEW_GROUPS:
        group = molmo_data.get(str(n_views), {}); row = {"categoria": cat, "object_id": oid, "n_views": n_views}
        try:
            pred = estimate_axis_confidence_proxy(group.get("points_by_image", {}), group.get("images_sent", []), fov_deg, image_size)
            ang  = angular_error_deg_signed(np.array(pred["direction"]), t_dir)
            dist = point_to_line_distance(np.array(pred["origin"]), t_orig, t_dir)
            row.update({"angular_error_deg": ang, "translation_error": dist})
            for t in ANGULAR_THRESHOLDS: row[f"precision_{t}deg"] = int(ang < t)
        except ValueError:
            row["angular_error_deg"] = 90.0
        rows_D.append(row)

summary_D = summarize_results(rows_D, f"EXP-D: Confianza proxy longitud×margen (AssemblyHands-X 2025)")


### EXP-E — Downweighting por residual de reproyección post-hoc
*(Hess-Flores et al., eScholarship; Zhang et al. arXiv:2008.01258)*

**Motivación**: después de una primera triangulación, las vistas con mayor
residual de reproyección son las que más degradan la solución. En lugar de
filtrarlas por ángulo *antes* de inferencia (lo que sesgaría las métricas),
se hace una **segunda pasada** que las downpesa según su residual.

**Flujo**:
1. Primera triangulación (EXP-A: ponderada por longitud 2D).
2. Calcular residual de cada vista: ángulo entre el plano de interpretación
   estimado y el eje triangulado (debería ser 0 si el eje está en el plano).
3. Peso actualizado: \(w_i \leftarrow w_i \cdot \exp(-\lambda \cdot r_i)\)
   donde \(r_i\) es el residual en radianes y \(\lambda\) es un factor de
   escala (típicamente 5–20).
4. Segunda triangulación con los pesos actualizados.

**Referencias**:  
Hess-Flores M. et al. (2014). *Uncertainty, Baseline, and Noise Analysis for
L1 Error-Based Multi-View Triangulation*. eScholarship.  
Zhang et al. (2020). arXiv:2008.01258.


In [ ]:
# ── EXP-E: Iterative residual-based downweighting ────────────────────────────
# Referencias: Hess-Flores et al. eScholarship; Zhang et al. arXiv:2008.01258

RESIDUAL_LAMBDA  = 10.0   # exponential decay factor for residual weighting
RESIDUAL_N_ITER  =  2     # number of reweighting iterations


def estimate_axis_residual_reweight(points_by_image: dict, images_sent: list,
                                     fov_deg: float, image_size: int,
                                     n_iter: int = RESIDUAL_N_ITER,
                                     lam: float = RESIDUAL_LAMBDA):
    """
    EXP-E: iterative reweighted interpretation-plane triangulation.
    Initial weights = 2D segment lengths.
    After each iteration, weight_i *= exp(-lambda * residual_i), where
    residual_i = |sin(angle between axis direction and plane i)| = |n_i . d|.
    """
    # Collect view data
    view_data = []
    for img_idx_str, pts in points_by_image.items():
        pair = widest_pair(pts)
        if pair is None:
            continue
        p_a, p_b = pair
        cam = images_sent[int(img_idx_str)]
        C, d_a = ray_dir_for_point(p_a["x"], p_a["y"], cam["R"], cam["T"], fov_deg, image_size)
        _, d_b  = ray_dir_for_point(p_b["x"], p_b["y"], cam["R"], cam["T"], fov_deg, image_size)
        n = interpretation_plane_normal(d_a, d_b)
        if n is None:
            continue
        seg_len = ((p_a["x"] - p_b["x"])**2 + (p_a["y"] - p_b["y"])**2) ** 0.5
        view_data.append({"C": C, "n": n, "w": max(seg_len, 1e-3)})

    if len(view_data) < 2:
        raise ValueError(f"EXP-E: need >=2 valid views, got {len(view_data)}")

    centers = [v["C"] for v in view_data]
    normals = [v["n"] for v in view_data]
    weights = [v["w"] for v in view_data]

    for iteration in range(n_iter):
        point, direction = triangulate_line_weighted(centers, normals, weights)
        # residual_i = |n_i . d| (sine of angle between axis and plane normal;
        # should be 0 for a perfect plane containing the axis)
        residuals = [abs(float(np.dot(n, direction))) for n in normals]
        weights   = [w * np.exp(-lam * r) for w, r in zip(weights, residuals)]
        total_w   = sum(weights)
        if total_w < 1e-9:
            break   # all weights collapsed — stop early
        weights = [w / total_w for w in weights]

    return {"direction": direction.tolist(), "origin": point.tolist(),
            "n_views_used": len(view_data), "method": "residual_reweight"}


# ── Run EXP-E ─────────────────────────────────────────────────────────────────
rows_E = []
for cat, oid in OBJECT_LIST:
    render_dir = SANDBOX_RENDERS / SYMMETRY_TYPE / oid / str(SIZE) / LIGHTING
    manifest   = json.load(open(render_dir / "manifest.json")) if (render_dir / "manifest.json").exists() else {}
    fov_deg    = manifest.get("fov", DEFAULT_FOV); image_size = manifest.get("image_size", SIZE)
    json_path  = render_dir / f"molmo_multiview_{EXPERIMENT_ID}.json"
    molmo_data = json.load(open(json_path))
    gt         = parse_true_label(SANDBOX_OBJECTS_DIR / f"{oid}.txt")
    t_dir      = np.array(gt["direction"]); t_orig = np.array(gt["origin"])

    for n_views in VIEW_GROUPS:
        group = molmo_data.get(str(n_views), {}); row = {"categoria": cat, "object_id": oid, "n_views": n_views}
        try:
            pred = estimate_axis_residual_reweight(group.get("points_by_image", {}), group.get("images_sent", []), fov_deg, image_size)
            ang  = angular_error_deg_signed(np.array(pred["direction"]), t_dir)
            dist = point_to_line_distance(np.array(pred["origin"]), t_orig, t_dir)
            row.update({"angular_error_deg": ang, "translation_error": dist})
            for t in ANGULAR_THRESHOLDS: row[f"precision_{t}deg"] = int(ang < t)
        except ValueError:
            row["angular_error_deg"] = 90.0
        rows_E.append(row)

summary_E = summarize_results(rows_E, f"EXP-E: Iterative residual reweighting (Hess-Flores / Zhang 2020)")


### EXP-F — Pipeline iterativo 3-tier para simetría planar
*(Propuesto en esta tesis; criterio de parada por SDE geométrico)*

**Motivación**: los datos muestran que con n_views=14, el sistema predice
~1.9 planos/objeto pero el GT tiene ~1.18. Los falsos positivos destruyen la
precisión. El pipeline iterativo detecta planos secuencialmente, removiendo
inliers antes de buscar el siguiente, con un SDE como criterio de parada.

**Flujo por objeto**:
1. **Tier 1**: triangular el mejor plano de todos los puntos.
   Calcular SDE_geom (distancia media de puntos al plano ÷ bbox_diag).
   Si SDE_geom < `sde_threshold`: añadir plano, continuar. Si no: parar.
2. **Tier 2**: remover puntos inliers del plano 1 (distancia al plano < `inlier_thr`).
   Si quedan ≥ `min_residual_pts` puntos con ≥ 2 vistas válidas:
   triangular → evaluar SDE → añadir si OK.
3. **Tier 3**: ídem con residuos de planos 1 y 2.

> **Nota**: Esta celda requiere que el prompt de inferencia haya pedido puntos de simetría
> **planar** (no axial). Cambiar `SYMMETRY_TYPE = "plane_sym"` y correr la inferencia
> con el prompt de plano antes de ejecutar esta celda.

**Referencia metodológica**: Gao et al. (2021). *PRS-Net*. IEEE TVCG 27(6):3007–3018.
DOI: 10.1109/TVCG.2020.3003823  (estrategia de eliminación de planos duplicados adaptada).


In [ ]:
# ── EXP-F: Iterative 3-tier planar symmetry detection ───────────────────────
# Referencia: Gao et al. PRS-Net, IEEE TVCG 2021

SDE_THRESHOLD_F   = 0.02   # máximo SDE normalizado para aceptar un plano
INLIER_THR_F      = 0.05   # distancia al plano para considerar un punto inlier (normalizada por bbox)
MIN_RESIDUAL_PTS  = 3      # mínimo de puntos en vistas residuales para intentar el siguiente tier

# ── Helper: bbox diagonal estimation from 3D points (no mesh needed)
def bbox_diagonal_from_points(pts_3d: list) -> float:
    if len(pts_3d) < 2:
        return 1.0
    arr = np.array(pts_3d)
    return float(np.linalg.norm(arr.max(axis=0) - arr.min(axis=0))) or 1.0


# ── Helper: SDE geométrico sin malla (proxy: mean point-to-plane distance)
def sde_proxy(plane_normal: np.ndarray, plane_origin: np.ndarray,
              points_3d: list) -> float:
    """
    Proxy SDE: mean |point . n - origin . n| / bbox_diagonal.
    Used as stopping criterion — not the same as the full SDE_ref that uses
    the mesh, but correlates with it for reasonably sampled point sets.
    """
    if not points_3d:
        return 1.0
    pts = np.array(points_3d)
    n = plane_normal / np.linalg.norm(plane_normal)
    d0 = float(np.dot(plane_origin, n))
    dists = np.abs(pts @ n - d0)
    bbox  = bbox_diagonal_from_points(points_3d)
    return float(dists.mean() / bbox)


# ── Helper: triangulate plane from interpretation planes
#    (same SVD machinery as triangulate_line but for the plane case:
#     the 3D plane's normal is the eigenvector of max eigenvalue of M,
#     and the origin is the centroid of all 3D anchor estimates)
def triangulate_plane_from_lines(centers: list, normals_perp: list,
                                  weights: list) -> tuple[np.ndarray, np.ndarray]:
    """
    For plane symmetry:  interpretation planes are formed from the 2D trace line.
    The symmetry plane's normal lies along the null space of all interpretation plane normals
    — i.e., it is the eigenvector of MAX eigenvalue of sum_i w_i n_i n_i^T.
    (Dual to the axis case where we use MIN eigenvalue.)
    """
    N  = np.asarray(normals_perp, dtype=np.float64)
    C  = np.asarray(centers,      dtype=np.float64)
    w  = np.asarray(weights,      dtype=np.float64); w = w / w.sum()
    M  = (N * w[:, None]).T @ N
    _, _, Vt = np.linalg.svd(M)
    normal = Vt[0]   # MAX eigenvalue for plane (dual to axis)
    normal /= np.linalg.norm(normal)
    # Origin: weighted centroid of camera centers projected onto plane
    origin = np.average(C, weights=w, axis=0)
    return origin, normal


def estimate_planes_3tier(points_by_image: dict, images_sent: list,
                           fov_deg: float, image_size: int,
                           render_dir: Path, image_cache: dict | None = None):
    """
    EXP-F: iterative 3-tier plane detection with SDE-based stopping.
    Returns a list of up to 3 plane dicts: [{normal, origin, sde, tier}, ...].
    """
    if image_cache is None:
        image_cache = {}

    # Filter off-object points first
    pts_by_img = filter_points_on_object(
        points_by_image, images_sent, render_dir, min_points=2, image_cache=image_cache)

    # Collect all valid 3D anchor estimates for bbox (proxy, no mesh)
    all_rays = []
    for img_idx_str, pts in pts_by_img.items():
        pair = widest_pair(pts)
        if pair is None: continue
        p_a, p_b = pair
        cam = images_sent[int(img_idx_str)]
        C, d_a = ray_dir_for_point(p_a["x"], p_a["y"], cam["R"], cam["T"], fov_deg, image_size)
        _, d_b  = ray_dir_for_point(p_b["x"], p_b["y"], cam["R"], cam["T"], fov_deg, image_size)
        all_rays.append((C, d_a, d_b,
                         (p_a["x"] - p_b["x"])**2 + (p_a["y"] - p_b["y"])**2,
                         img_idx_str))

    # Use midpoint triangulation for 3D anchors (bbox proxy)
    anchor_pts_3d = [midpoint_triangulate_pair(C, d_a, C, d_b) for C, d_a, d_b, _, _ in all_rays]
    bbox_diag = bbox_diagonal_from_points(anchor_pts_3d) if len(anchor_pts_3d) >= 2 else 1.0

    current_pts_by_img = dict(pts_by_img)   # mutable copy for tier iteration
    results = []

    for tier in range(1, 4):
        # Build interpretation planes for current point set
        centers, normals, weights = [], [], []
        for img_idx_str, pts in current_pts_by_img.items():
            pair = widest_pair(pts)
            if pair is None: continue
            p_a, p_b = pair
            cam = images_sent[int(img_idx_str)]
            C, d_a = ray_dir_for_point(p_a["x"], p_a["y"], cam["R"], cam["T"], fov_deg, image_size)
            _, d_b  = ray_dir_for_point(p_b["x"], p_b["y"], cam["R"], cam["T"], fov_deg, image_size)
            n = interpretation_plane_normal(d_a, d_b)
            if n is None: continue
            seg_len = ((p_a["x"] - p_b["x"])**2 + (p_a["y"] - p_b["y"])**2) ** 0.5
            centers.append(C); normals.append(n); weights.append(max(seg_len, 1e-3))

        if len(normals) < 2:
            break  # not enough views for this tier

        origin, normal = triangulate_plane_from_lines(centers, normals, weights)

        # Proxy SDE using anchor 3D points
        sde = sde_proxy(normal, origin, anchor_pts_3d)

        if sde > SDE_THRESHOLD_F:
            break  # plane not geometrically consistent — stop

        results.append({"normal": normal.tolist(), "origin": origin.tolist(),
                         "sde": round(sde, 6), "tier": tier})

        # Remove inlier points (2D points whose ray is close to the plane)
        # Proxy: points whose camera-to-anchor direction aligns with the plane
        inlier_thr_abs = INLIER_THR_F * bbox_diag
        new_pts_by_img = {}
        for img_idx_str, pts in current_pts_by_img.items():
            cam = images_sent[int(img_idx_str)]
            kept = []
            for p in pts:
                C, d = ray_dir_for_point(p["x"], p["y"], cam["R"], cam["T"], fov_deg, image_size)
                # Approximate 3D location: project ray to plane
                n_hat = normal / np.linalg.norm(normal)
                denom = np.dot(d, n_hat)
                if abs(denom) > 1e-6:
                    t_int = (np.dot(origin, n_hat) - np.dot(C, n_hat)) / denom
                    if t_int > 0:
                        pt_3d = C + t_int * d
                        dist_to_plane = abs(float(np.dot(pt_3d - origin, n_hat)))
                        if dist_to_plane > inlier_thr_abs:
                            kept.append(p)
                    else:
                        kept.append(p)
                else:
                    kept.append(p)
            if len(kept) >= 2:
                new_pts_by_img[img_idx_str] = kept
        current_pts_by_img = new_pts_by_img

        total_residual_pts = sum(len(v) for v in current_pts_by_img.values())
        if total_residual_pts < MIN_RESIDUAL_PTS * 2:
            break  # not enough residual points for the next tier

    return results if results else None


# ── Evaluacion EXP-F (requiere SYMMETRY_TYPE = "plane_sym" y GT planares) ────
# Si el notebook esta en modo axial, esta celda muestra el flujo pero
# retorna resultados vacios -- cambiar SYMMETRY_TYPE para activarla.

rows_F = []
for cat, oid in OBJECT_LIST:
    render_dir = SANDBOX_RENDERS / SYMMETRY_TYPE / oid / str(SIZE) / LIGHTING
    manifest   = json.load(open(render_dir / "manifest.json")) if (render_dir / "manifest.json").exists() else {}
    fov_deg    = manifest.get("fov", DEFAULT_FOV); image_size = manifest.get("image_size", SIZE)
    json_path  = render_dir / f"molmo_multiview_{EXPERIMENT_ID}.json"
    if not json_path.exists():
        continue
    molmo_data = json.load(open(json_path))
    img_cache  = {}

    # Load GT planes
    gt_txt = SANDBOX_OBJECTS_DIR / f"{oid}.txt"
    gt_lines = [l.strip() for l in gt_txt.read_text().splitlines() if l.strip() and l.startswith("plane")]
    gt_planes = []
    for line in gt_lines:
        parts = line.split()
        n_gt   = np.array([float(x) for x in parts[1:4]]); n_gt /= np.linalg.norm(n_gt)
        o_gt   = np.array([float(x) for x in parts[4:7]])
        gt_planes.append({"normal": n_gt, "origin": o_gt})

    for n_views in VIEW_GROUPS:
        group = molmo_data.get(str(n_views), {})
        row   = {"categoria": cat, "object_id": oid, "n_views": n_views, "n_gt_planes": len(gt_planes)}
        try:
            pred_planes = estimate_planes_3tier(
                group.get("points_by_image", {}), group.get("images_sent", []),
                fov_deg, image_size, render_dir, image_cache=img_cache)
            if pred_planes is None:
                pred_planes = []
            row["n_pred_planes"] = len(pred_planes)
            # Greedy matching (PRS-Net convention)
            matched_gt = set()
            tp = 0
            for pp in pred_planes:
                pn = np.array(pp["normal"])
                for gi, gp in enumerate(gt_planes):
                    if gi in matched_gt: continue
                    ang = angular_error_deg_signed(pn, gp["normal"])
                    if ang < 15.0:     # 15° threshold for F1 matching
                        tp += 1; matched_gt.add(gi); break
            prec = tp / len(pred_planes) if pred_planes else 0.0
            rec  = tp / len(gt_planes)   if gt_planes   else 0.0
            f1   = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0
            row.update({"precision": round(prec, 4), "recall": round(rec, 4), "f1": round(f1, 4)})
        except (ValueError, Exception) as e:
            row.update({"precision": 0.0, "recall": 0.0, "f1": 0.0, "n_pred_planes": 0})
        rows_F.append(row)

df_F = pd.DataFrame(rows_F)
if not df_F.empty and "f1" in df_F.columns:
    print("\n=== EXP-F: 3-Tier planar pipeline ===")
    print(df_F.groupby("n_views")[["precision","recall","f1","n_pred_planes"]].mean().round(4).to_string())
else:
    print("[EXP-F] No data — ensure SYMMETRY_TYPE='plane_sym' and plane inference has been run.")


---
## 11. Comparación de ablaciones

Tabla resumen de todos los experimentos por n_views para comparación directa.
Permite identificar qué cambio individual produce mayor mejora en AUC y P@10°.


In [ ]:
# ── Tabla comparativa de todas las ablaciones ────────────────────────────────

def make_summary_row(label: str, rows: list[dict]) -> dict:
    df = pd.DataFrame(rows)
    df["angular_error_deg"] = df["angular_error_deg"].fillna(90.0)
    out = {"experimento": label}
    for nv in VIEW_GROUPS:
        sub = df[df["n_views"] == nv]
        if sub.empty: continue
        out[f"AUC@45°_nv{nv}"]  = round(auc_angular(sub["angular_error_deg"].tolist()), 4)
        out[f"P@10°_nv{nv}"]    = round(sub.get(f"precision_10deg", pd.Series([0])).mean(), 4)
        out[f"median_err_nv{nv}"] = round(sub["angular_error_deg"].median(), 2)
    return out

comparison_rows = [
    make_summary_row("Baseline (original)",  rows),          # rows from Section 8
    make_summary_row("EXP-A: Weighted (Bartoli 2005)", rows_A),
    make_summary_row("EXP-B: PtD (Wu 2021)", rows_B),
    make_summary_row("EXP-C: RANSAC 2D (Recker 2013)", rows_C),
    make_summary_row("EXP-D: Conf proxy (AssemblyHands 2025)", rows_D),
    make_summary_row("EXP-E: Residual reweight (Hess-Flores)", rows_E),
]

df_comparison = pd.DataFrame(comparison_rows).set_index("experimento")
print("\n=== Comparación de ablaciones (eje de simetría axial) ===")
display(df_comparison)

# Highlight best per column
best_mask = df_comparison == df_comparison.max()
print("\n--- Mejor por métrica ---")
for col in df_comparison.columns:
    best_exp = df_comparison[col].idxmax()
    best_val = df_comparison[col].max()
    print(f"  {col}: {best_exp}  ({best_val})")


---
## 12. Diagnóstico por objeto — ¿Qué experimento mejora qué objetos?

Identificar si las mejoras son uniformes o si benefician solo ciertos casos.


In [ ]:
# ── Per-object: best experiment per object ───────────────────────────────────
N_VIEWS_DIAG = 14   # n_views para el diagnóstico por objeto

def extract_errors(rows: list[dict], n_views: int) -> dict:
    """Return {object_id: angular_error_deg} for a given n_views."""
    df = pd.DataFrame(rows)
    df["angular_error_deg"] = df["angular_error_deg"].fillna(90.0)
    sub = df[df["n_views"] == n_views]
    return dict(zip(sub["object_id"], sub["angular_error_deg"]))

exp_errors = {
    "Baseline":  extract_errors(rows,   N_VIEWS_DIAG),
    "EXP-A":     extract_errors(rows_A, N_VIEWS_DIAG),
    "EXP-B":     extract_errors(rows_B, N_VIEWS_DIAG),
    "EXP-C":     extract_errors(rows_C, N_VIEWS_DIAG),
    "EXP-D":     extract_errors(rows_D, N_VIEWS_DIAG),
    "EXP-E":     extract_errors(rows_E, N_VIEWS_DIAG),
}

all_oids = sorted(set(oid for cat, oid in OBJECT_LIST))
df_per_obj = pd.DataFrame({exp: [errors.get(oid, 90.0) for oid in all_oids]
                            for exp, errors in exp_errors.items()},
                           index=all_oids)
df_per_obj["best_exp"]  = df_per_obj.idxmin(axis=1)
df_per_obj["best_err"]  = df_per_obj.min(axis=1).round(2)
df_per_obj["baseline"]  = df_per_obj["Baseline"].round(2)
df_per_obj["delta"]     = (df_per_obj["best_err"] - df_per_obj["baseline"]).round(2)

print(f"\n=== Diagnóstico por objeto (n_views={N_VIEWS_DIAG}) ===")
display(df_per_obj[["baseline", "best_exp", "best_err", "delta"]].sort_values("delta"))

print(f"\n--- Frecuencia de mejor experimento ---")
print(df_per_obj["best_exp"].value_counts().to_string())

# Visualization: error distribution comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, nv in zip(axes, [6, 14]):
    for exp_name, rows_exp in [
        ("Baseline", rows), ("EXP-A", rows_A), ("EXP-B", rows_B),
        ("EXP-C", rows_C), ("EXP-D", rows_D), ("EXP-E", rows_E)]:
        errors = [r.get("angular_error_deg", 90.0) for r in rows_exp if r["n_views"] == nv]
        thetas = np.linspace(0, 45, 200)
        precisions = [np.mean(np.array(errors) <= t) for t in thetas]
        ax.plot(thetas, precisions, label=exp_name)
    ax.set_xlabel("Umbral angular (°)"); ax.set_ylabel("Fracción de objetos bajo umbral")
    ax.set_title(f"Curva Precision@θ — n_views={nv}")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("ablation_precision_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: ablation_precision_curves.png")
